# Demo 2: SQL basics with Chinook

This demo practices core SQL clauses on the Chinook tables imported into DuckDB. Run demo 1 first to set up the database.

## Step 0: Connect to database

In [1]:
%load_ext sql
%sql duckdb:///demo_chinook.duckdb

Connecting to 'duckdb:///demo_chinook.duckdb'

## Step 1: Basic SELECT + LIMIT

In [2]:
%%sql
SELECT TrackId, Name, Milliseconds
FROM tracks
LIMIT 5;

Running query in 'duckdb:///demo_chinook.duckdb'

TrackId,Name,Milliseconds
1,For Those About To Rock (We Salute You),343719
2,Balls to the Wall,342562
3,Fast As a Shark,230619
4,Restless and Wild,252051
5,Princess of the Dawn,375418


Checkpoint: you should see 5 tracks with IDs and durations.

Example output:

| TrackId | Name | Milliseconds |
| --- | --- | --- |
| 1 | For Those About To Rock (We Salute You) | 343719 |
| 2 | Balls to the Wall | 342562 |
| 3 | Fast As a Shark | 230619 |

## Step 2: WHERE + ORDER BY

In [3]:
%%sql
SELECT Name, UnitPrice, Milliseconds
FROM tracks
WHERE UnitPrice >= 0.99
ORDER BY UnitPrice DESC, Milliseconds DESC
LIMIT 10;

Running query in 'duckdb:///demo_chinook.duckdb'

Name,UnitPrice,Milliseconds
Occupation / Precipice,1.99,5286953
Through a Looking Glass,1.99,5088838
"Greetings from Earth, Pt. 1",1.99,2960293
The Man With Nine Lives,1.99,2956998
"Battlestar Galactica, Pt. 2",1.99,2956081
"Battlestar Galactica, Pt. 1",1.99,2952702
Murder On the Rising Star,1.99,2935894
"Battlestar Galactica, Pt. 3",1.99,2927802
Take the Celestra,1.99,2927677
Fire In Space,1.99,2926593


Checkpoint: higher-priced tracks appear first.

## Step 3: DISTINCT values

In [4]:
%%sql
SELECT DISTINCT Country
FROM customers
ORDER BY Country;

Running query in 'duckdb:///demo_chinook.duckdb'

Country
Argentina
Australia
Austria
Belgium
Brazil
Canada
Chile
Czech Republic
Denmark
Finland


Checkpoint: one row per country.

## Step 3b: DISTINCT vs GROUP BY

In [5]:
%%sql
SELECT DISTINCT BillingCountry
FROM invoices
ORDER BY BillingCountry;

Running query in 'duckdb:///demo_chinook.duckdb'

BillingCountry
Argentina
Australia
Austria
Belgium
Brazil
Canada
Chile
Czech Republic
Denmark
Finland


In [6]:
%%sql
SELECT BillingCountry, COUNT(*) AS invoice_count
FROM invoices
GROUP BY BillingCountry
ORDER BY BillingCountry;

Running query in 'duckdb:///demo_chinook.duckdb'

BillingCountry,invoice_count
Argentina,7
Australia,7
Austria,7
Belgium,7
Brazil,35
Canada,56
Chile,7
Czech Republic,14
Denmark,7
Finland,7


Checkpoint: the first query returns unique countries, the second adds counts.

## Step 3c: COUNT(DISTINCT ...)

In [7]:
%%sql
SELECT COUNT(DISTINCT CustomerId) AS unique_customers
FROM invoices;

Running query in 'duckdb:///demo_chinook.duckdb'

unique_customers
59


Checkpoint: a single row with a unique customer count.

## Step 3d: CASE WHEN for categories

In [8]:
%%sql
SELECT InvoiceId,
    Total,
    CASE
        WHEN Total >= 10 THEN 'high'
        WHEN Total >= 5 THEN 'medium'
        ELSE 'low'
    END AS spend_bucket
FROM invoices
ORDER BY Total DESC
LIMIT 10;

Running query in 'duckdb:///demo_chinook.duckdb'

InvoiceId,Total,spend_bucket
404,25.86,high
299,23.86,high
96,21.86,high
194,21.86,high
89,18.86,high
201,18.86,high
88,17.91,high
306,16.86,high
313,16.86,high
208,15.86,high


Checkpoint: results include a `spend_bucket` column.

## Step 3e: LIKE vs ILIKE

In [9]:
%%sql
SELECT TrackId, Name
FROM tracks
WHERE Name ILIKE '%love%'
LIMIT 10;

Running query in 'duckdb:///demo_chinook.duckdb'

TrackId,Name
24,Love In An Elevator
56,"Love, Hate, Love"
195,Let Me Love You Baby
335,My Love
341,The Girl I Love She Got Long Black Wavy Hair
345,Whole Lotta Love
413,Loverman
440,Love Gun
444,Do You Love Me
449,Calling Dr. Love


Checkpoint: track names containing "love" (case-insensitive).

## Step 4: GROUP BY + aggregates

In [10]:
%%sql
SELECT BillingCountry, COUNT(*) AS invoice_count, ROUND(AVG(Total), 2) AS avg_total
FROM invoices
GROUP BY BillingCountry
ORDER BY invoice_count DESC;

Running query in 'duckdb:///demo_chinook.duckdb'

BillingCountry,invoice_count,avg_total
USA,91,5.75
Canada,56,5.43
Brazil,35,5.43
France,35,5.57
Germany,28,5.59
United Kingdom,21,5.37
Czech Republic,14,6.45
Portugal,14,5.52
India,13,5.79
Argentina,7,5.37


Example output:

| BillingCountry | invoice_count | avg_total |
| --- | --- | --- |
| USA | 91 | 5.75 |
| Canada | 56 | 5.42 |
| Brazil | 35 | 5.32 |

## Step 4b: Plot invoice counts by country (Altair)

In [11]:
%%sql invoice_counts <<
SELECT BillingCountry, COUNT(*) AS invoice_count
FROM invoices
GROUP BY BillingCountry
ORDER BY invoice_count DESC

Running query in 'duckdb:///demo_chinook.duckdb'

In [12]:
import altair as alt

chart = (
    alt.Chart(invoice_counts)
    .mark_bar()
    .encode(
        x=alt.X("BillingCountry:N", sort="-y", title="Billing country"),
        y=alt.Y("invoice_count:Q", title="Invoice count"),
        tooltip=["BillingCountry", "invoice_count"]
    )
)

chart

/Users/christopher/Documents/datasci_223/.venv/lib/python3.12/site-packages/altair/vegalite/v6/api.py:303: UserWarning: data of type <class 'sql.run.resultset.ResultSet'> not recognized
  warnings.warn(f"data of type {type(data)} not recognized", stacklevel=1)


ValueError: BillingCountry encoding field is specified without a type; the type cannot be automatically inferred because the data is not specified as a pandas.DataFrame.

alt.Chart(...)

Checkpoint: you should see a bar chart with the largest countries on the left.

## Step 5: HAVING for group filters

In [13]:
%%sql
SELECT BillingCountry, COUNT(*) AS invoice_count
FROM invoices
GROUP BY BillingCountry
HAVING COUNT(*) >= 5
ORDER BY invoice_count DESC;

Running query in 'duckdb:///demo_chinook.duckdb'

BillingCountry,invoice_count
USA,91
Canada,56
Brazil,35
France,35
Germany,28
United Kingdom,21
Czech Republic,14
Portugal,14
India,13
Denmark,7


Checkpoint: the result should include only countries with at least 5 invoices.